In [11]:
import cv2
import numpy as np
import os
from pathlib import Path

In [24]:
def crop_obb_from_corners(image, corners_px):
    """
    Crops an oriented bounding box (OBB) from an image using perspective transform,
    given the four corner points.

    Args:
        image (np.ndarray): The input image.
        corners_px (np.ndarray): A 4x2 numpy array of the OBB's corner coordinates
                                 in pixels, e.g., [[x1,y1], [x2,y2], [x3,y3], [x4,y4]].
                                 The order is assumed to be sequential around the polygon
                                 (e.g., p1, p2, p3, p4 clockwise or counter-clockwise).
                                 p1-p2 will define the width of the output crop.
                                 p2-p3 will define the height of the output crop.

    Returns:
        np.ndarray: The cropped image area, or None if cropping fails.
    """
    try:
        # Ensure corners are float32 for cv2.getPerspectiveTransform
        src_pts = corners_px.astype(np.float32)

        # Calculate the width and height of the output image
        # Width: distance between the first two points (p1, p2)
        # Height: distance between the second and third points (p2, p3)
        
        # Vector from p1 to p2
        edge_vec1 = src_pts[1] - src_pts[0]
        # Vector from p2 to p3
        edge_vec2 = src_pts[2] - src_pts[1]
        
        target_w = np.linalg.norm(edge_vec1)
        target_h = np.linalg.norm(edge_vec2)
        
        # HERE IS THE LIKELY PROBLEM if corners_px are still normalized:
        # If target_w is, for example, 0.4 pixels (because it was calculated from normalized coords),
        # int(round(0.4)) becomes 0.
        target_w_int = int(round(target_w))
        target_h_int = int(round(target_h))

        if target_w_int <= 0 or target_h_int <= 0:
            print(f"Warning: Invalid OBB dimensions calculated (w={target_w_int}, h={target_h_int}). Skipping crop.")
            print(f"Corners: {corners_px.tolist()}")
            return None

        # Define the destination points for the perspective transform
        # This will be a new image of size (target_w_int, target_h_int)
        # The source points src_pts[0], src_pts[1], src_pts[2], src_pts[3]
        # will be mapped to these destination points respectively.
        dst_pts = np.array([
            [0, 0],                            # src_pts[0] (e.g., top-left)
            [target_w_int - 1, 0],             # src_pts[1] (e.g., top-right)
            [target_w_int - 1, target_h_int - 1],# src_pts[2] (e.g., bottom-right)
            [0, target_h_int - 1]              # src_pts[3] (e.g., bottom-left)
        ], dtype="float32")

        # Get the perspective transformation matrix
        M = cv2.getPerspectiveTransform(src_pts, dst_pts)

        # Warp the image
        warped_image = cv2.warpPerspective(image, M, (target_w_int, target_h_int),
                                           flags=cv2.INTER_CUBIC, # Smoother interpolation
                                           borderMode=cv2.BORDER_REPLICATE) # How to fill outside pixels
        
        return warped_image

    except Exception as e:
        print(f"Error during cropping from corners: {e}")
        print(f"Corners: {corners_px.tolist() if corners_px is not None else 'N/A'}")
        return None

In [25]:
def process_image_and_labels(image_path, label_path, output_dir, class_names=None, min_confidence=0.0, coords_normalized=True):
    """
    Processes a single image and its OBB label file (4-corner format) to crop detections.
    Returns the number of crops successfully made.
    """
    # --- MODIFIED IMAGE READING ---
    try:
        # Use np.fromfile and cv2.imdecode to handle non-ASCII paths robustly
        n = np.fromfile(str(image_path), np.uint8) # Read file as bytes
        image = cv2.imdecode(n, cv2.IMREAD_COLOR) # Decode image from bytes
        if image is None:
            # This check is important because cv2.imdecode can also return None
            print(f"Error: cv2.imdecode failed for image {image_path}. The file might be corrupted or not a supported image format.")
            return 0
    except FileNotFoundError:
        print(f"Error: File not found at path {image_path}")
        return 0
    except Exception as e:
        print(f"Error: Could not read image {image_path} using np.fromfile/cv2.imdecode. Exception: {e}")
        return 0
    # --- END OF MODIFIED IMAGE READING ---

    img_h, img_w = image.shape[:2]
    image_basename = image_path.stem

    if not label_path.exists():
        print(f"Warning: Label file {label_path} not found for image {image_path}")
        return 0

    crops_made_for_this_image = 0
    with open(label_path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        parts = line.strip().split()
        try:
            # Expecting class_id + 8 coordinates (4 points * 2 coords) = 9 parts
            # Or class_id + 8 coordinates + confidence = 10 parts
            if not (len(parts) == 9 or len(parts) == 10):
                print(f"Warning: Skipping line {i+1} in {label_path} due to unexpected number of parts: {len(parts)}. Content: '{line.strip()}'")
                continue

            class_id = int(parts[0])
            
            # Extract 8 coordinate values and reshape to 4x2 array
            coords_values = [float(p) for p in parts[1:9]]
            # corners will be [[x1,y1], [x2,y2], [x3,y3], [x4,y4]]
            corners = np.array(coords_values).reshape((4, 2))

            # confidence = 1.0 # Default if not present
            # if len(parts) == 10: # Confidence is present
            #     try:
            #         confidence = float(parts[9])
            #     except ValueError:
            #         print(f"Warning: Could not parse confidence from '{parts[9]}' in {label_path}, line {i+1}. Using 1.0.")

        except (ValueError, IndexError) as e:
            print(f"Error parsing line {i+1} in {label_path}: '{line.strip()}'. Error: {e}")
            continue
        
        # if confidence < min_confidence:
        #     continue

        corners_px = np.copy(corners) # Make a copy to modify if normalized
        if coords_normalized:
            corners_px[:, 0] *= img_w  # Denormalize x coordinates
            corners_px[:, 1] *= img_h  # Denormalize y coordinates

        cropped_obb = crop_obb_from_corners(image, corners_px)

        if cropped_obb is not None:
            class_str = str(class_id)
            if class_names and 0 <= class_id < len(class_names):
                class_str = class_names[class_id].replace(" ", "_") # Make filename friendly

            # crop_filename = f"{image_basename}_obj{i:03d}_cls{class_str}_conf{confidence:.2f}.png"
            crop_filename = f"{image_basename}_obj{i:03d}_cls{class_str}.png"
            output_filepath = output_dir / crop_filename
            
            try:
                cv2.imwrite(str(output_filepath), cropped_obb)
                crops_made_for_this_image += 1
            except Exception as e:
                print(f"Error saving cropped image {output_filepath}: {e}")
    
    return crops_made_for_this_image

In [26]:
image_dir = Path("datasets/fst__annotations/images/val/")
label_dir = Path("datasets/fst__annotations/labels/val/")
output_dir = Path("check_cutting")

# if not image_dir.is_dir():
#     print(f"Error: Image directory {image_dir} not found.")
#     return
# if not label_dir.is_dir():
#     print(f"Error: Label directory {label_dir} not found.")
#     return

output_dir.mkdir(parents=True, exist_ok=True)

class_names = None
# if args.class_names_file:
#     try:
#         with open(args.class_names_file, 'r') as f:
#             class_names = [line.strip() for line in f.readlines() if line.strip()]
#         print(f"Loaded {len(class_names)} class names.")
#     except FileNotFoundError:
#         print(f"Warning: Class names file {args.class_names_file} not found. Using class IDs in filenames.")

image_extensions = [f".{ext.strip()}" for ext in "png,jpg,jpeg,bmp,tif,tiff".split(',')]

processed_images_count = 0
total_crops_count = 0

image_files_to_process = [
    f for f in image_dir.iterdir() if f.is_file() and f.suffix.lower() in image_extensions
]

if not image_files_to_process:
    print(f"No images found in {image_dir} with specified extensions: {', '.join(image_extensions)}")
    
print(f"Found {len(image_files_to_process)} images to process.")
# if args.coords_normalized:
#     print("Interpreting label coordinates as NORMALIZED (0-1 range).")
# else:
#     print("Interpreting label coordinates as ABSOLUTE PIXELS.")

for image_file in image_files_to_process:
    label_file = label_dir / f"{image_file.stem}.txt"
    
    if label_file.exists():
        # print(f"Processing: {image_file.name} with {label_file.name}") # Uncomment for verbose per-file logging
        num_crops_generated = process_image_and_labels(
            image_file, 
            label_file, 
            output_dir, 
            class_names, 
        )
        total_crops_count += num_crops_generated
        # Count as processed if label file exists, even if no crops met criteria (unless label file is empty)
        if os.path.getsize(label_file) > 0:
             processed_images_count +=1
        elif num_crops_generated > 0: # Should be covered by above but as a fallback
             processed_images_count +=1
        else: # Label file exists but is empty
             print(f"Label file {label_file.name} for {image_file.name} is empty.")

    # else:
    #     print(f"Skipping {image_file.name}: No corresponding label file {label_file.name} found.")

print(f"\nProcessing complete.")
print(f"Processed {processed_images_count} images that had corresponding label files.")
print(f"Generated {total_crops_count} crops in {output_dir}.")

Found 599 images to process.

Processing complete.
Processed 23 images that had corresponding label files.
Generated 115 crops in check_cutting.


In [3]:
import cv2
import numpy as np
import os
import math
from pathlib import Path

In [4]:
def crop_oriented_bounding_box(image, cx, cy, w, h, angle_degrees):
    """
    Crops an oriented bounding box (OBB) from an image using perspective transform.

    Args:
        image (np.ndarray): The input image.
        cx (float): Center x-coordinate of the OBB in image pixels.
        cy (float): Center y-coordinate of the OBB in image pixels.
        w (float): Width of the OBB in image pixels.
        h (float): Height of the OBB in image pixels.
        angle_degrees (float): Angle of the OBB in degrees.
                               Assumed to be CCW from positive x-axis to OBB's width axis.

    Returns:
        np.ndarray: The cropped image area, or None if cropping fails.
    """
    try:
        target_w = int(round(w))
        target_h = int(round(h))

        if target_w <= 0 or target_h <= 0:
            print(f"Warning: Invalid OBB dimensions (w={target_w}, h={target_h}). Skipping crop.")
            return None

        # Convert angle to radians for trigonometric functions
        angle_rad = math.radians(angle_degrees)
        cos_a = math.cos(angle_rad)
        sin_a = math.sin(angle_rad)

        # Define the 4 corners of the OBB in its own coordinate system (centered at origin)
        # Order: Top-Left, Top-Right, Bottom-Right, Bottom-Left
        half_w, half_h = w / 2.0, h / 2.0
        rect_coords = np.array([
            [-half_w, -half_h],  # Top-Left
            [ half_w, -half_h],  # Top-Right
            [ half_w,  half_h],  # Bottom-Right
            [-half_w,  half_h]   # Bottom-Left
        ])

        # Rotate the corners
        # Rotation matrix: [[cos_a, -sin_a], [sin_a, cos_a]]
        # For points (x,y): x' = x*cos_a - y*sin_a, y' = x*sin_a + y*cos_a
        rotated_coords = np.zeros_like(rect_coords)
        for i, (x, y) in enumerate(rect_coords):
            rotated_coords[i, 0] = x * cos_a - y * sin_a
            rotated_coords[i, 1] = x * sin_a + y * cos_a
        
        # Translate corners to image coordinates
        src_pts = rotated_coords + np.array([cx, cy])
        src_pts = src_pts.astype(np.float32)

        # Define the destination points for the perspective transform
        # This will be a new image of size (target_w, target_h)
        dst_pts = np.array([
            [0, 0],                         # Top-Left
            [target_w - 1, 0],              # Top-Right
            [target_w - 1, target_h - 1],   # Bottom-Right
            [0, target_h - 1]               # Bottom-Left
        ], dtype="float32")

        # Get the perspective transformation matrix
        M = cv2.getPerspectiveTransform(src_pts, dst_pts)

        # Warp the image
        warped_image = cv2.warpPerspective(image, M, (target_w, target_h),
                                           flags=cv2.INTER_CUBIC, # Smoother interpolation
                                           borderMode=cv2.BORDER_REPLICATE) # How to fill outside pixels
        
        return warped_image

    except Exception as e:
        print(f"Error during cropping: {e}")
        print(f"Params: cx={cx}, cy={cy}, w={w}, h={h}, angle={angle_degrees}")
        print(f"src_pts: {src_pts if 'src_pts' in locals() else 'N/A'}")
        print(f"dst_pts: {dst_pts if 'dst_pts' in locals() else 'N/A'}")
        return None

In [5]:
def process_image_and_labels(image_path, label_path, output_dir, class_names=None, min_confidence=0.0):
    """
    Processes a single image and its YOLO OBB label file to crop detections.
    """
    image = cv2.imread(str(image_path))
    if image is None:
        print(f"Error: Could not read image {image_path}")
        return

    img_h, img_w = image.shape[:2]
    image_basename = image_path.stem

    if not label_path.exists():
        print(f"Warning: Label file {label_path} not found for image {image_path}")
        return

    with open(label_path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        parts = line.strip().split()
        try:
            class_id = int(parts[0])
            cx_norm = float(parts[1])
            cy_norm = float(parts[2])
            w_norm = float(parts[3])
            h_norm = float(parts[4])
            angle_rad = float(parts[5]) # Assumed to be in radians
            
            confidence = 1.0 # Default if not present
            if len(parts) > 6: # Check if confidence is present
                try:
                    confidence = float(parts[6])
                except ValueError:
                    print(f"Warning: Could not parse confidence from '{parts[6]}' in {label_path}, line {i+1}. Using 1.0.")


        except (ValueError, IndexError) as e:
            print(f"Error parsing line {i+1} in {label_path}: '{line.strip()}'. Error: {e}")
            continue
        
        if confidence < min_confidence:
            continue

        # Denormalize coordinates
        cx_px = cx_norm * img_w
        cy_px = cy_norm * img_h
        w_px = w_norm * img_w
        h_px = h_norm * img_h

        # Convert angle from radians (YOLO OBB output) to degrees (for our function)
        # Common convention: angle_rad is CCW from positive x-axis to OBB's width axis
        angle_deg = math.degrees(angle_rad)

        # --- IMPORTANT ANGLE NOTE ---
        # The `crop_oriented_bounding_box` function expects `angle_deg` to be the
        # CCW rotation of the OBB's width (w_px) axis from the image's positive x-axis.
        # If your YOLO-OBB model outputs angles with a different definition
        # (e.g., different range, CW, relative to y-axis, or always for the longest side),
        # you MUST adjust `angle_deg` here before passing it to the crop function.
        # For example, some OpenCV functions (like cv2.RotatedRect) have specific angle
        # conventions (e.g., angle in (-90, 0] degrees). Our crop function is more direct.

        cropped_obb = crop_oriented_bounding_box(image, cx_px, cy_px, w_px, h_px, angle_deg)

        if cropped_obb is not None:
            class_str = str(class_id)
            if class_names and 0 <= class_id < len(class_names):
                class_str = class_names[class_id].replace(" ", "_") # Make filename friendly

            # Create a unique filename for the crop
            crop_filename = f"{image_basename}_obj{i:03d}_cls{class_str}_conf{confidence:.2f}.png"
            output_filepath = output_dir / crop_filename
            
            try:
                cv2.imwrite(str(output_filepath), cropped_obb)
                # print(f"Saved: {output_filepath}")
            except Exception as e:
                print(f"Error saving cropped image {output_filepath}: {e}")

In [10]:
image_dir = Path("datasets/fst__annotations/images/val/")
label_dir = Path("datasets/fst__annotations/labels/val/")
output_dir = Path("check_cutting")
min_confidence = 0.35

if not image_dir.is_dir():
    print(f"Error: Image directory {image_dir} not found.")
if not label_dir.is_dir():
    print(f"Error: Label directory {label_dir} not found.")

output_dir.mkdir(parents=True, exist_ok=True)

class_names = None
# if args.class_names_file:
#     try:
#         with open(args.class_names_file, 'r') as f:
#             class_names = [line.strip() for line in f.readlines() if line.strip()]
#         print(f"Loaded {len(class_names)} class names.")
#     except FileNotFoundError:
#         print(f"Warning: Class names file {args.class_names_file} not found. Using class IDs.")

# image_extensions = [f".{ext.strip()}" for ext in args.image_ext.split(',')]
image_extensions = [f".{ext.strip()}" for ext in "png,jpg,jpeg,bmp,tif,tiff".split(',')]

processed_images = 0
total_crops = 0

for image_file in image_dir.iterdir():
    if image_file.is_file() and image_file.suffix.lower() in image_extensions:
        label_file = label_dir / f"{image_file.stem}.txt"
        if label_file.exists():
            print(f"Processing: {image_file.name} with {label_file.name}")
            # Count initial crops to see how many were made for this image
            current_crops_in_output = len(list(output_dir.glob(f"{image_file.stem}_obj*.png")))
            process_image_and_labels(image_file, label_file, output_dir, class_names, min_confidence)
            new_crops = len(list(output_dir.glob(f"{image_file.stem}_obj*.png"))) - current_crops_in_output
            total_crops += new_crops
            processed_images +=1
        # else:
        #     print(f"Skipping {image_file.name}: No corresponding label file {label_file.name} found.")

print(f"\nProcessing complete. Processed {processed_images} images and generated {total_crops} crops in {output_dir}.")

Processing: skv.142801_керн1-5.jpg with skv.142801_керн1-5.txt
Error: Could not read image datasets\fst__annotations\images\val\skv.142801_керн1-5.jpg
Processing: skv.142801_керн11-15.jpg with skv.142801_керн11-15.txt
Error: Could not read image datasets\fst__annotations\images\val\skv.142801_керн11-15.jpg
Processing: skv.142801_керн16-20.jpg with skv.142801_керн16-20.txt
Error: Could not read image datasets\fst__annotations\images\val\skv.142801_керн16-20.jpg
Processing: skv.142801_керн21-25.jpg with skv.142801_керн21-25.txt
Error: Could not read image datasets\fst__annotations\images\val\skv.142801_керн21-25.jpg
Processing: skv.142801_керн26-30.jpg with skv.142801_керн26-30.txt
Error: Could not read image datasets\fst__annotations\images\val\skv.142801_керн26-30.jpg
Processing: skv.142801_керн31-35.jpg with skv.142801_керн31-35.txt
Error: Could not read image datasets\fst__annotations\images\val\skv.142801_керн31-35.jpg
Processing: skv.142801_керн36-40.jpg with skv.142801_керн36-40.t